# Bahdanau (Additive) Attention

In [1]:
import tensorflow as tf

class BahdanauAttention(tf.keras.layers.Layer):
    """
    Additive attention (Bahdanau et al. 2015)
    Call signature: context, weights = layer(query, values)
    - query: decoder hidden state, shape (batch, hidden)  OR (batch, 1, hidden)
    - values: encoder outputs, shape (batch, seq_len, hidden)
    Returns:
    - context: (batch, hidden)  (weighted sum of values)
    - weights: (batch, seq_len) (attention weights)
    """
    def __init__(self, units):
        super().__init__()
        self.w1 = tf.keras.layers.Dense(units)  # for query
        self.w2 = tf.keras.layers.Dense(units)  # for values
        self.v  = tf.keras.layers.Dense(1)      # to compute score

    def call(self, query, values, mask=None):
        # Ensure query shape is (batch, 1, hidden)
        if len(query.shape) == 2:
            query = tf.expand_dims(query, axis=1)

        # score shape -> (batch, seq_len, 1)
        score = self.v(tf.nn.tanh(self.w1(query) + self.w2(values)))

        # remove last dim -> (batch, seq_len)
        score = tf.squeeze(score, axis=-1)

        if mask is not None:
            # mask should be (batch, seq_len) with 1 for real tokens
            score += (1.0 - tf.cast(mask, tf.float32)) * -1e9

        weights = tf.nn.softmax(score, axis=1)  # (batch, seq_len)

        # context: weighted sum over values -> (batch, hidden)
        context = tf.matmul(tf.expand_dims(weights, 1), values)
        context = tf.squeeze(context, axis=1)

        return context, weights


# Luong (Multiplicative) Attention — dot and general

In [2]:
class LuongAttention(tf.keras.layers.Layer):
    """
    Multiplicative attention (Luong et al. 2015).
    modes: "dot" (simple), "general" (learned linear transform)
    Call: context, weights = layer(query, values)
    """
    def __init__(self, mode="dot"):
        super().__init__()
        assert mode in ("dot", "general")
        self.mode = mode
        if self.mode == "general":
            self.W = tf.keras.layers.Dense(units=None)  # unitless Dense -> will infer units at build

    def build(self, input_shape):
        # input_shape isn't strictly needed because Dense will adapt, but we accept default behavior
        super().build(input_shape)

    def call(self, query, values, mask=None):
        # query: (batch, hidden) or (batch, 1, hidden)
        if len(query.shape) == 2:
            query = tf.expand_dims(query, 1)  # (batch,1,hidden)

        if self.mode == "dot":
            # score = values @ query^T over hidden dim
            # values: (batch, seq_len, hidden), query: (batch,1,hidden)
            # -> inner product across last dim: (batch, seq_len)
            score = tf.reduce_sum(values * query, axis=-1)
        else:  # "general"
            # apply W to query then dot
            q_trans = self.W(query)  # (batch,1,hidden)
            score = tf.reduce_sum(values * q_trans, axis=-1)

        if mask is not None:
            score += (1.0 - tf.cast(mask, tf.float32)) * -1e9

        weights = tf.nn.softmax(score, axis=1)  # (batch, seq_len)
        context = tf.matmul(tf.expand_dims(weights, 1), values)  # (batch,1,hidden)
        context = tf.squeeze(context, axis=1)  # (batch, hidden)
        return context, weights


# How to use in a simple decoder loop (teacher forcing)

In [3]:
# Example encoder outputs & initial decoder state (toy shapes)
batch = 32
seq_len = 10
enc_hidden = 64
tgt_vocab_size = 500
embedding_dim = 32
dec_units = 64
timesteps = 7  # target length

encoder_outputs = tf.random.normal([batch, seq_len, enc_hidden])  # encoder return_sequences
encoder_mask = tf.ones([batch, seq_len], dtype=tf.int32)          # (1 for real tokens)

# decoder initial state (h,c) if using LSTM:
decoder_h = tf.random.normal([batch, dec_units])
decoder_c = tf.random.normal([batch, dec_units])

# layers
embedding = tf.keras.layers.Embedding(tgt_vocab_size, embedding_dim)
decoder_cell = tf.keras.layers.LSTMCell(dec_units)
bahd = BahdanauAttention(units=dec_units)  # or LuongAttention("dot")

# example target input sequence (teacher forcing)
target_inputs = tf.random.uniform([batch, timesteps], maxval=tgt_vocab_size, dtype=tf.int32)

# run loop
outputs = []
state = [decoder_h, decoder_c]
for t in range(timesteps):
    x_t = embedding(target_inputs[:, t])  # (batch, embedding_dim)

    # attention - use hidden state h (for LSTM it's state[0])
    context, attn_w = bahd(state[0], encoder_outputs, mask=encoder_mask)

    # combine input and context (simple concat)
    cell_input = tf.concat([tf.expand_dims(context, 1), tf.expand_dims(x_t, 1)], axis=-1)
    cell_input = tf.squeeze(cell_input, axis=1)  # (batch, context+embed)

    # one step of LSTMCell
    output, state = decoder_cell(cell_input, state)  # output shape (batch, dec_units)

    outputs.append(output)

# final outputs tensor
decoder_outputs = tf.stack(outputs, axis=1)  # (batch, timesteps, dec_units)
print("decoder_outputs shape:", decoder_outputs.shape)


decoder_outputs shape: (32, 7, 64)


# Quick unit test (verify shapes)

In [4]:
# Simple shape check
batch = 4
seq_len = 6
hidden = 8

values = tf.random.normal([batch, seq_len, hidden])
query = tf.random.normal([batch, hidden])

bahd = BahdanauAttention(units=16)
ctx, w = bahd(query, values)
print("Bahdanau context:", ctx.shape)  # (batch, hidden)
print("Bahdanau weights:", w.shape)    # (batch, seq_len)

luong = LuongAttention(mode="dot")
ctx2, w2 = luong(query, values)
print("Luong context:", ctx2.shape)    # (batch, hidden)
print("Luong weights:", w2.shape)      # (batch, seq_len)


Bahdanau context: (4, 8)
Bahdanau weights: (4, 6)
Luong context: (4, 8)
Luong weights: (4, 6)
